In [ ]:
# Exercise 4. (When was Tom in the class?)
# Consider the example ”When was Tom in the class?” from the course ”AI and ML”. The
# knowledge base KB is given by the logical term:
# KB=(M∨T)∧[(M ∧¬W)∨(¬M ∧W)]∧(T ∨W)
# where M,T,W are propositional variables. Verfify that KB |= T, i.e. KB entails T by
#    a) considering the truth tables of KB and T,
#    b) comparing the models of KB and T,
#    c) applying the function satisfiable() to the term KB ∨ ¬T

In [3]:
# a)
from sympy import symbols
from sympy.logic.boolalg import truth_table

M, T, W = symbols('M T W')

KB = (M | T) & ((M & ~W) | (~M & W)) & (T | W)

print("M     T     W  |  KB   |  T")
print("-" * 35)
for (m, t, w), kb_val in truth_table(KB, [M, T, W]):
    print(f"{int(bool(m))}     {int(bool(t))}     {int(bool(w))}  |   {int(bool(kb_val))}   |  {int(bool(t))}")

M     T     W  |  KB   |  T
-----------------------------------
0     0     0  |   0   |  0
0     0     1  |   0   |  0
0     1     0  |   0   |  1
0     1     1  |   1   |  1
1     0     0  |   0   |  0
1     0     1  |   0   |  0
1     1     0  |   1   |  1
1     1     1  |   0   |  1


In [4]:
#b)
from sympy.logic.inference import satisfiable

# Collect all models of KB
models_KB = list(satisfiable(KB, all_models=True))
print("Models of KB:")
for m in models_KB:
    print(f"  {m}")

# Collect all models of T (trivially: any assignment where T=True)
models_T = list(satisfiable(T, all_models=True))
print("\nModels of T:")
for m in models_T:
    print(f"  {m}")

# Check subset relationship
# Every model of KB must have T=True
entailed = all(m.get(T, False) for m in models_KB)
print(f"\nKB |= T : {entailed}")

Models of KB:
  {M: True, W: False, T: True}
  {M: False, W: True, T: True}

Models of T:
  {T: True}

KB |= T : True


In [5]:
# c)
# If KB ∧ ¬T is unsatisfiable → KB |= T
result = satisfiable(KB & ~T)

if result == False:
    print("KB ∧ ¬T is UNSATISFIABLE  →  KB |= T  ✓")
else:
    print(f"KB ∧ ¬T is SATISFIABLE  →  KB does NOT entail T")
    print(f"Counterexample: {result}")
    

KB ∧ ¬T is UNSATISFIABLE  →  KB |= T  ✓


In [6]:
# Exercise 5.
#Write two versions of a Python function entails(KB,φ), which returns True if KB |= φ,
#otherwise False by
#    a) comparing the models of KB and φ,
#    b) applying the function satisfiable() to the term KB ∨ ¬φ

In [7]:
# a)
from sympy import symbols
from sympy.logic.boolalg import And
from sympy.logic.inference import satisfiable
from sympy.logic.boolalg import truth_table

def entails_models(KB, phi):
    """KB |= phi iff every model of KB is also a model of phi."""
    
    models_KB = list(satisfiable(KB, all_models=True))
    
    # Edge case: KB is a tautology → {False: False} sentinel
    if models_KB == [{False: False}]:
        # Every valuation satisfies KB, so check if phi is also a tautology
        models_phi = list(satisfiable(phi, all_models=True))
        return models_phi == [{False: False}]
    
    # For each model of KB, check it also satisfies phi
    for model in models_KB:
        # Substitute the model's values into phi
        phi_val = phi.subs(list(model.items()))
        if not bool(phi_val):
            return False  # Found a counterexample
    
    return True


# ── Test with Exercise 4 ─────────────────────────────────────────────
M, T, W = symbols('M T W')
KB = (M | T) & ((M & ~W) | (~M & W)) & (T | W)

print(f"KB |= T  : {entails_models(KB, T)}")   # Expected: True
print(f"KB |= M  : {entails_models(KB, M)}")   # Expected: False
print(f"KB |= W  : {entails_models(KB, W)}")   # Expected: False

KB |= T  : True
KB |= M  : False
KB |= W  : False


In [8]:
#b)
def entails_sat(KB, phi):
    """KB |= phi iff KB ∧ ¬phi is unsatisfiable."""
    
    result = satisfiable(KB & ~phi)
    return result == False  # Unsatisfiable → entailment holds


# ── Test with Exercise 4 ─────────────────────────────────────────────
print(f"KB |= T  : {entails_sat(KB, T)}")   # Expected: True
print(f"KB |= M  : {entails_sat(KB, M)}")   # Expected: False
print(f"KB |= W  : {entails_sat(KB, W)}")   # Expected: False

KB |= T  : True
KB |= M  : False
KB |= W  : False


In [9]:
# cross checking both 
print("Comparing both implementations:")
print(f"  entails_models(KB, T) = {entails_models(KB, T)}")
print(f"  entails_sat(KB, T)    = {entails_sat(KB, T)}")

print(f"  entails_models(KB, M) = {entails_models(KB, M)}")
print(f"  entails_sat(KB, M)    = {entails_sat(KB, M)}")

Comparing both implementations:
  entails_models(KB, T) = True
  entails_sat(KB, T)    = True
  entails_models(KB, M) = False
  entails_sat(KB, M)    = False


In [10]:
# Exercise 6.
#a) Uselambdasandlist comprehension to define functions Tupel2(BS), Tupel3(BS),
# Tupel4(BS) of 2-, 3- and 4-Tupel over a base set BS of symbols.
#b) Define a recursive function Tupel(n,BS) which generates all n-Tupels over a
# basis set BS of symbols.
#c) Define functions myTruthTable(term) and myModels(term) which computes a
# truth table respectively all models of a logical term.

In [11]:
# a)
# Using list comprehension inside a lambda
Tupel2 = lambda BS: [(x1, x2) 
                     for x1 in BS 
                     for x2 in BS]

Tupel3 = lambda BS: [(x1, x2, x3) 
                     for x1 in BS 
                     for x2 in BS 
                     for x3 in BS]

Tupel4 = lambda BS: [(x1, x2, x3, x4) 
                     for x1 in BS 
                     for x2 in BS 
                     for x3 in BS 
                     for x4 in BS]

# Test
BS = [0, 1]
print(f"2-Tuples: {Tupel2(BS)}")
print(f"3-Tuples: {Tupel3(BS)}")
print(f"Number of 4-Tuples: {len(Tupel4(BS))}")  # Should be 16

2-Tuples: [(0, 0), (0, 1), (1, 0), (1, 1)]
3-Tuples: [(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]
Number of 4-Tuples: 16


In [12]:
#b)
def Tupel(n, BS):
    if n == 1:
        return [(x,) for x in BS]
    else:
        # For each smaller tuple, extend it with each element of BS
        return [t + (x,) for t in Tupel(n-1, BS) for x in BS]

# Test
BS = [0, 1]
print(f"Tupel(2, BS): {Tupel(2, BS)}")
print(f"Tupel(3, BS): {Tupel(3, BS)}")
print(f"Tupel(4, BS) count: {len(Tupel(4, BS))}")  # 16

# Verify Tupel matches the lambdas from a)
print(f"\nTupel(2) == Tupel2: {Tupel(2, BS) == Tupel2(BS)}")
print(f"Tupel(3) == Tupel3: {Tupel(3, BS) == Tupel3(BS)}")
print(f"Tupel(4) == Tupel4: {Tupel(4, BS) == Tupel4(BS)}")

Tupel(2, BS): [(0, 0), (0, 1), (1, 0), (1, 1)]
Tupel(3, BS): [(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]
Tupel(4, BS) count: 16

Tupel(2) == Tupel2: True
Tupel(3) == Tupel3: True
Tupel(4) == Tupel4: True


In [13]:
# c)
from sympy import symbols
from sympy.logic.boolalg import And, Or, Not

def myTruthTable(term):
    """Compute truth table of a SymPy boolean term manually."""
    
    # Extract the free variables from the term
    variables = sorted(term.free_symbols, key=lambda s: s.name)
    n = len(variables)
    
    # All combinations of True/False for n variables
    valuations = Tupel(n, [False, True])
    
    print("  " + "  ".join(str(v) for v in variables) + "  |  Result")
    print("-" * (6 * n + 12))
    
    table = []
    for vals in valuations:
        substitution = list(zip(variables, vals))
        result = bool(term.subs(substitution))
        row = dict(zip(variables, vals))
        row['result'] = result
        table.append(row)
        print("  " + "  ".join(str(int(v)) for v in vals) + f"  |  {int(result)}")
    
    return table


def myModels(term):
    """Return all models (satisfying valuations) of a SymPy boolean term."""
    
    variables = sorted(term.free_symbols, key=lambda s: s.name)
    n = len(variables)
    valuations = Tupel(n, [False, True])
    
    models = []
    for vals in valuations:
        substitution = list(zip(variables, vals))
        result = bool(term.subs(substitution))
        if result:
            models.append(dict(zip(variables, vals)))
    
    return models


# ── Test with KB from Exercise 4 ─────────────────────────────────────
M, T, W = symbols('M T W')
KB = (M | T) & ((M & ~W) | (~M & W)) & (T | W)

print("Truth table of KB:")
myTruthTable(KB)

print("\nModels of KB:")
for model in myModels(KB):
    print(f"  {model}")

Truth table of KB:
  M  T  W  |  Result
------------------------------
  0  0  0  |  0
  0  0  1  |  0
  0  1  0  |  0
  0  1  1  |  1
  1  0  0  |  0
  1  0  1  |  0
  1  1  0  |  1
  1  1  1  |  0

Models of KB:
  {M: False, T: True, W: True}
  {M: True, T: True, W: False}
